In [1]:
import anywidget

import ipywidgets

import sys

print("Python:", sys.executable)

print("anywidget:", anywidget.__version__)

print("ipywidgets:", ipywidgets.__version__)

Python: /Applications/Blender.app/Contents/Resources/5.1/python/bin/python3.13
anywidget: 0.11.0
ipywidgets: 8.1.8


In [2]:
# %% 1) Make deps + the gesture_widget package importable in Blender's Python
import subprocess
import sys

REPO = "/Users/jan-hendrik/projects/caputre_motion"  # <-- adjust if you moved it

# anywidget etc. must live in BLENDER's bundled Python (this kernel), not your
# system Python. sys.executable here is Blender's python; install once.
for pkg in ("anywidget", "traitlets", "ipywidgets"):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

src = f"{REPO}/src_widget"
if src not in sys.path:
    sys.path.insert(0, src)

from gesture_widget import GestureRecognizerWidget  # noqa: E402

print("gesture_widget ready in Blender's Python")


# %% 2) Show the widget, then click "Start Camera" in the output and allow webcam
w = GestureRecognizerWidget()
w.sync_interval_ms = 33  # push landmarks to Python at ~30 Hz
w


gesture_widget ready in Blender's Python


In [3]:


# %% 3) One cube per hand landmark = all 21 MediaPipe joints, driven by bpy
#        (main-thread safe)
import bpy  # noqa: E402

SCALE = 5.0   # world units spanned by the camera frame
CUBE = 0.08   # half-size of each landmark cube

# {traitlet name on `w`: Blender object name}, covering all 21 landmarks
# from the MediaPipe hand-landmark model bundle.
LANDMARKS = {
    "wrist": "LM_Wrist",
    "thumb_cmc": "LM_Thumb_CMC",
    "thumb_mcp": "LM_Thumb_MCP",
    "thumb_ip": "LM_Thumb_IP",
    "thumb_tip": "LM_Thumb_Tip",
    "index_finger_mcp": "LM_Index_MCP",
    "index_finger_pip": "LM_Index_PIP",
    "index_finger_dip": "LM_Index_DIP",
    "index_finger_tip": "LM_Index_Tip",
    "middle_finger_mcp": "LM_Middle_MCP",
    "middle_finger_pip": "LM_Middle_PIP",
    "middle_finger_dip": "LM_Middle_DIP",
    "middle_finger_tip": "LM_Middle_Tip",
    "ring_finger_mcp": "LM_Ring_MCP",
    "ring_finger_pip": "LM_Ring_PIP",
    "ring_finger_dip": "LM_Ring_DIP",
    "ring_finger_tip": "LM_Ring_Tip",
    "pinky_mcp": "LM_Pinky_MCP",
    "pinky_pip": "LM_Pinky_PIP",
    "pinky_dip": "LM_Pinky_DIP",
    "pinky_tip": "LM_Pinky_Tip",
}


In [4]:

cubes = {}
for trait, obj_name in LANDMARKS.items():
    obj = bpy.data.objects.get(obj_name)
    if obj is None:
        import bmesh

        mesh = bpy.data.meshes.new(obj_name)
        bm = bmesh.new()
        bmesh.ops.create_cube(bm, size=CUBE * 2)
        bm.to_mesh(mesh)
        bm.free()
        obj = bpy.data.objects.new(obj_name, mesh)
        bpy.context.scene.collection.objects.link(obj)
    cubes[trait] = obj


def _update_landmarks():
    # Reading traitlets (plain Python attributes) and doing bpy work is safe here
    # because timers run on Blender's main thread.
    for trait, obj in cubes.items():
        p = getattr(w, trait)  # [x, y, z] normalized 0..1, or [] when no hand
        if p:
            x, y, z = p
            # image space (x right, y DOWN) -> Blender (x right, z UP, y depth)
            obj.location = ((x - 0.5) * SCALE, z * SCALE, (0.5 - y) * SCALE)
    return 0.033  # seconds until next call (~30 Hz). Return None to stop.


if bpy.app.timers.is_registered(_update_landmarks):
    bpy.app.timers.unregister(_update_landmarks)
bpy.app.timers.register(_update_landmarks)

print(f"Timer running — {len(cubes)} cubes now track all hand landmarks.")


# %% 4) Stop driving Blender
# bpy.app.timers.unregister(_update_landmarks)
# w.stop()  # turn the camera off


Timer running — 21 cubes now track all hand landmarks.
